In [1]:
!pip install requests beautifulsoup4 pandas

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time

In [3]:
# --- CONFIGURACIÓN INICIAL ---
print("🚀 INICIANDO PROCESO INTEGRADO...")
datos_hockey_final = []

def extraer_dato(fila, clase_css):
    elemento = fila.find(class_=clase_css)
    return elemento.text.strip() if elemento else "0"

🚀 INICIANDO PROCESO INTEGRADO...


In [4]:
# --- FASE 1: SCRAPPING COMPLETO (24 PÁGINAS) ---
for numero_pagina in range(1, 25):
    url = f"https://www.scrapethissite.com/pages/forms/?page_num={numero_pagina}"
    print(f"-> Procesando página {numero_pagina}/24...")
    
    try:
        respuesta = requests.get(url, timeout=15)
        soup = BeautifulSoup(respuesta.text, 'html.parser')
        filas = soup.find_all('tr', class_='team')
        
        for fila in filas:
            datos_hockey_final.append({
                "Equipo": extraer_dato(fila, 'name'),
                "Año": extraer_dato(fila, 'year'),
                "Victorias": extraer_dato(fila, 'wins'),
                "Derrotas": extraer_dato(fila, 'losses'),
                "Derrotas_OT": extraer_dato(fila, 'ot-losses'),
                "Porcentaje_Victorias": extraer_dato(fila, 'pct'),
                "Goles_Favor": extraer_dato(fila, 'gf'),
                "Goles_Contra": extraer_dato(fila, 'ga')
            })
        time.sleep(1) # Pausa ética
    except Exception as e:
        print(f"Error en página {numero_pagina}: {e}")

-> Procesando página 1/24...
-> Procesando página 2/24...
-> Procesando página 3/24...
-> Procesando página 4/24...
-> Procesando página 5/24...
-> Procesando página 6/24...
-> Procesando página 7/24...
-> Procesando página 8/24...
-> Procesando página 9/24...
-> Procesando página 10/24...
-> Procesando página 11/24...
-> Procesando página 12/24...
-> Procesando página 13/24...
-> Procesando página 14/24...
-> Procesando página 15/24...
-> Procesando página 16/24...
-> Procesando página 17/24...
-> Procesando página 18/24...
-> Procesando página 19/24...
-> Procesando página 20/24...
-> Procesando página 21/24...
-> Procesando página 22/24...
-> Procesando página 23/24...
-> Procesando página 24/24...


In [5]:
# --- FASE 2: CREACIÓN DEL FRAME Y LIMPIEZA ---
df_completo = pd.DataFrame(datos_hockey_final)

# Convertimos a números para que el Frame sea útil
cols_num = ['Año', 'Victorias', 'Derrotas', 'Derrotas_OT', 'Goles_Favor', 'Goles_Contra']
for col in cols_num:
    df_completo[col] = pd.to_numeric(df_completo[col], errors='coerce')

In [6]:
# --- FASE 3: ALMACENAMIENTO (CSV Y SQLITE) ---
# Guardamos en CSV para Excel
df_completo.to_csv('historial_hockey_master.csv', index=False, encoding='utf-8-sig')

# Guardamos en SQLite para la Base de Datos
conexion = sqlite3.connect('historial_hockey_master.db')
df_completo.to_sql('tabla_equipos', conexion, if_exists='replace', index=False)
conexion.close()

print("\n✅ PROCESO FINALIZADO.")
print("📁 Archivo CSV generado: historial_hockey_master.csv")
print("🗄️ Base de Datos actualizada: historial_hockey_master.db")
print("📊 Mostrando Frame interactivo abajo:\n")


✅ PROCESO FINALIZADO.
📁 Archivo CSV generado: historial_hockey_master.csv
🗄️ Base de Datos actualizada: historial_hockey_master.db
📊 Mostrando Frame interactivo abajo:



In [7]:
# --- FASE 4: VISUALIZACIÓN DEL FRAME ---
# Esto activará el visor interactivo de VS Code
df_completo

,Equipo,Año,Victorias,Derrotas,Derrotas_OT,Porcentaje_Victorias,Goles_Favor,Goles_Contra
0,Boston Bruins,1990,44,24,NaN,0.55,299,264
1,Buffalo Sabres,1990,31,30,NaN,0.388,292,278
2,Calgary Flames,1990,46,26,NaN,0.575,344,263
3,Chicago Blackhawks,1990,49,23,NaN,0.613,284,211
4,Detroit Red Wings,1990,34,38,NaN,0.425,273,298
...,...,...,...,...,...,...,...,...
577,Tampa Bay Lightning,2011,38,36,8.0,0.463,235,281
578,Toronto Maple Leafs,2011,35,37,10.0,0.427,231,264
579,Vancouver Canucks,2011,51,22,9.0,0.622,249,198
580,Washington Capitals,2011,42,32,8.0,0.512,222,230
